In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from expecto import get_spectrum

In [ ]:
T_phot = 2600 * u.K
T_cool = 2300 * u.K
T_hot = 5000 * u.K

phot = get_spectrum(T_phot.value, 5, cache=True)
cool = get_spectrum(T_cool.value, 5, cache=True)
hot = get_spectrum(T_hot.value, 5, cache=True)

In [ ]:
for component in [phot, cool, hot]:
    plt.loglog(component.wavelength, component.flux)

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse, FancyArrowPatch

In [ ]:
from specutils import Spectrum1D

def spectral_binning(y, all_x, all_y):
    """
    Spectral binning via trapezoidal approximation.
    """
    min_ind = np.argwhere(all_y == y[0])[0, 0]
    max_ind = np.argwhere(all_y == y[-1])[0, 0]
    if max_ind > min_ind and y.shape == all_x[min_ind:max_ind + 1].shape:
        return np.trapz(y, all_x[min_ind:max_ind + 1]) / (all_x[max_ind] - all_x[min_ind])
    return np.nan

def bin_spectrum(spectrum, bins=None, log=True, min=None, max=None, **kwargs):
    """
    
    Bin a spectrum, with log-spaced frequency bins.

    Parameters
    ----------
    spectrum : 
    log : bool
        If true, compute bin edges based on the log base 10 of
        the frequency.
    bins : int or ~numpy.ndarray
        Number of bins, or the bin edges

    Returns
    -------
    new_spectrum : 
    """
    from scipy.stats import binned_statistic
    
    nirspec_wl_range = (spectrum.wavelength > min) & (spectrum.wavelength < max)
    
    wavelength = spectrum.wavelength[nirspec_wl_range]
    flux = spectrum.flux[nirspec_wl_range]

    if log:
        wl_axis = np.log10(wavelength.to(u.um).value)
    else:
        wl_axis = wavelength.to(u.um).value

    # Bin the power spectrum:
    bs = binned_statistic(
        wl_axis, flux.value,
        statistic=lambda y: spectral_binning(
            y, all_x=wl_axis, all_y=flux.value
        ),
        bins=bins
    )
    if log:
        wl_bins = 10 ** (
            0.5 * (bs.bin_edges[1:] + bs.bin_edges[:-1])
        ) * u.um
    else:
        wl_bins = (
            0.5 * (bs.bin_edges[1:] + bs.bin_edges[:-1])
        ) * u.um
    nans = np.isnan(bs.statistic)
    interp_fluxes = bs.statistic.copy()
    interp_fluxes[nans] = np.interp(wl_bins[nans], wl_bins[~nans], bs.statistic[~nans])
    return Spectrum1D(flux=interp_fluxes * flux.unit, spectral_axis=wl_bins)

In [ ]:
from scipy.ndimage import gaussian_filter1d

water_wl, water_opacity = np.loadtxt('water_opacity.txt', unpack=True)
water_smoothed = gaussian_filter1d(water_opacity, 10)
water_resid = (water_smoothed - 5e-3)
sign_flips = np.argwhere(np.sign(water_resid[1:]) != np.sign(water_resid[:-1]))[:, 0]

TiO_wl, TiO_opacity = np.loadtxt('TiO_opacity.txt', unpack=True)
TiO_smoothed = gaussian_filter1d(TiO_opacity, 10)
TiO_fit = np.polyval(np.polyfit(TiO_wl - TiO_wl.mean(), np.log10(TiO_smoothed), 5), TiO_wl - TiO_wl.mean())
TiO_resid = (TiO_smoothed - 10 ** TiO_fit - 1e-1)
TiO_sign_flips = np.argwhere(
    (np.sign(TiO_resid[1:]) != np.sign(TiO_resid[:-1])) & 
    (TiO_wl[:-1] < 2)
)[:, 0]

VO_wl, VO_opacity = np.loadtxt('VO_opacity.txt', unpack=True)
VO_smoothed = gaussian_filter1d(VO_opacity, 10)
VO_fit = np.polyval(np.polyfit(VO_wl - VO_wl.mean(), np.log10(VO_smoothed), 5), VO_wl - VO_wl.mean())
VO_resid = (VO_smoothed - 10 ** VO_fit - 1e-1)
VO_sign_flips = np.argwhere(
    (np.sign(VO_resid[1:]) != np.sign(VO_resid[:-1])) & 
    (VO_wl[:-1] < 2)
)[:, 0]

In [ ]:
from matplotlib import lines

fig = plt.figure(figsize=(8, 3))

gs = GridSpec(1, 4)

ax_left = fig.add_subplot(gs[-1])
ax_left.axis('off')
theta = np.linspace(0, 2*np.pi, 50)
inc = np.radians(80)
rs = 0.7
x0 = 0.2
y0 = 0.5
x = x0 + rs * np.cos(theta)
y = rs * np.sin(theta)
z = y * np.cos(inc)

xx = np.linspace(-rs, rs, 1_000)
XX, YY = np.meshgrid(*[xx for i in range(2)])

ld = 1/100 * np.sqrt(rs**2 - XX**2 - YY**2) ** 1.1

ax_left.plot(x[y < 0], z[y < 0] + y0, 'k', ls='--', lw=0.8, alpha=0.5, zorder=10)

ax_left.set_title("TRAPPIST-1")

colors = "#cf4b42 #66040b #5ce1ff".split()

star = plt.Circle((x0, y0), rs, color='#c41f14', clip_on=True)
dynamic_range = ld[~np.isnan(ld)].max()
ax_left.pcolormesh(xx + x0, xx + y0, ld, alpha=1, vmin=0.1 * dynamic_range, vmax=dynamic_range, cmap=plt.cm.Reds_r, rasterized=True)

cool_spot_xy = (0.65, 0.2)
warm_spot_xy = (0.75, 0.6)
spot_cool = Ellipse(cool_spot_xy, 0.12, 0.2, angle=-45, color='#66040b')
spot_warm = Ellipse(warm_spot_xy, 0.07, 0.1, angle=8, color='#5ce1ff')

for feature in [spot_warm, spot_cool]:
    ax_left.add_patch(feature)
ax_left.set(
    xlim=[1, 0],
    ylim=[0, 1]
)

spectra = []

for s in [phot, cool, hot]:
    component = bin_spectrum(s, bins=3000, min=0.5*u.um, max=6*u.um, log=True)
    spectra.append(component)

ax_right = fig.add_subplot(gs[:-1])
labels = ['Photosphere, 2600 K', 'Cool region, 2300 K', 'Warm region, 5000 K']
for component, color, label in zip(spectra, colors, labels):
    if label.startswith('Phot'):
        lw_offset = 0.8
    else: 
        lw_offset = 0
    
    ax_right.loglog(component.wavelength.to(u.um), component.flux.to(u.erg/u.s/u.cm**3), color=color, lw=0.8 + lw_offset, label=label, rasterized=True)


ax_right.legend(title="Heterogeneous star:", frameon=False, alignment='left', fontsize=7)
xticks = np.arange(1, 6)
ax_right.set_ylim([1e10, 8e15])

fig.tight_layout()
transFigure = fig.transFigure.inverted()

for spot_xy, graphic_spec, color in zip(
    [[x0+rs, y0-0.1], cool_spot_xy, warm_spot_xy], spectra, colors
):
    xyB = [graphic_spec.wavelength.to(u.um).value[-1], graphic_spec.flux.to(u.erg/u.s/u.cm**3).value[-1]]
    coord1 = transFigure.transform(ax_left.transData.transform(spot_xy))
    coord2 = transFigure.transform(ax_right.transData.transform(xyB))
    line = lines.Line2D(
        (coord1[0], coord2[0]),  # xdata
        (coord1[1], coord2[1]),  # ydata
        transform=fig.transFigure,
        color=color, lw=0.8, ls='--', alpha=0.8
    )
    fig.lines.append(line)
    
species_color = 'gray'
ar1_y = 4.5e15
ar1_y_TiO = 1.5e15
ar1_y_VO = 0.5e15
fs = 13

for i in range(0, 6, 2):
    ax_right.plot([water_wl[sign_flips[i]], water_wl[sign_flips[i+1]]], [ar1_y, ar1_y], lw=0.6, color=species_color, zorder=10)

for ar1_i in range(1, len(TiO_sign_flips) - 1, 2):
    ax_right.plot([TiO_wl[TiO_sign_flips[ar1_i]], TiO_wl[TiO_sign_flips[ar1_i+1]]], [ar1_y_TiO, ar1_y_TiO], lw=0.6, color=species_color, zorder=10)

for ar1_i_vo in range(1, len(VO_sign_flips) - 1, 2):
    ax_right.plot([VO_wl[VO_sign_flips[ar1_i_vo]], VO_wl[VO_sign_flips[ar1_i_vo+1]]], [ar1_y_VO, ar1_y_VO], lw=0.6, color=species_color, zorder=10)
    
ax_right.annotate(
    "H$_2$O", (water_wl[sign_flips[-1]] + 0.05, ar1_y), 
    fontsize=fs - 5, color=species_color,
    va='center', ha='left'
)
ax_right.annotate(
    "TiO", (TiO_wl[TiO_sign_flips[ar1_i]] + 0.12, ar1_y_TiO), 
    fontsize=fs - 5, color=species_color,
    va='center', ha='left'
)
ax_right.annotate(
    "VO", (VO_wl[VO_sign_flips[ar1_i_vo]] + 0.1, ar1_y_VO), 
    fontsize=fs - 5, color=species_color,
    va='center', ha='left'
)
# plt.tick_params(axis='y', which='both', labelleft='off', labelright='on')
# ax_right.yaxis.tick_right() 
ax_right.set_ylabel(f'Flux [{component.flux.unit.to_string("latex")}]')

# ax_right.yaxis.set_label_position("right")
# ax_right.yaxis.tick_right()

ax_right.set(
    xticks=xticks,
    xticklabels=xticks,
    xlabel='Wavelength [$\mu$m]',
)
for sp in 'right top'.split():
    ax_right.spines[sp].set_visible(False)

fig.savefig('plots/schematic.pdf', bbox_inches='tight', dpi=300)